
O arquivo ./docs/roteiro-execucao.md  ->tem o roteiro de como executar o projeto (passo a passo)

In [0]:

fluxo de execução
1- ir no diretorio \notebook\ingestao
2- executar o arquivo 00_reset_ambiente = se quise zerar a base
2.1- executar o script 01_bronze para gerar as tabelas - brunto para bronze
2.2- executar o script 02_silver para gerar as tabelas - bronze para silver
2.3- executar o script 03_gold para gerar as tabelas - silver   

## ordem de execução

```
IMPORTANTE
    FLUXO PARA CARGA INICIAL

Ordem completa do pipeline agora:
    00_reset_ambiente → 
    01_bronze (training + scoring) → 
    02_silver → 
    03_gold → 
    notebooks/ead/01_eda_bronze + 
                  02_eda_silver.

    01_hipoteses_h1_h4.py

```



In [0]:
%m
```
Confirmar que a Gold está populada
Antes de tudo, confirme que a Gold existe e tem dado: spark.table(f'{catalog}.gold.fct_credit_profile').count() e spark.table(f'{catalog}.gold.dim_customer').count() devem retornar ~150.000. Se vier 0 ou erro, o problema é anterior a este passo — volte para 03_gold.py.

2
Rodar 01_hipoteses_h1_h4.py e ler as conclusões
Rode notebooks/ml/01_hipoteses_h1_h4.py com catalog=credito_dev. Confira no output: as duas linhas de ROC-AUC (XGBoost deve ficar perto de 0.85–0.87, coerente com a métrica-alvo do README) e os quatro blocos de conclusão (H1–H4) imprimindo 'Confirmada' ou 'Contrariada / Inconclusiva' com os números de evidência junto.

3
Confirmar a tabela gold_hypotheses_validation
Rode display(spark.table(f'{catalog}.gold.gold_hypotheses_validation')) — precisa ter exatamente 4 linhas (H1–H4), cada uma com status, metrica e evidencia preenchidos. Se a tabela não existir, o erro está na célula final de persistência — confira se o catalog do widget bate com o esperado.

4
Rodar 02_vies_threshold_roi.py (primeira vez treina o modelo)
Rode notebooks/ml/02_vies_threshold_roi.py com o mesmo catalog. Na primeira execução, como o alias '@champion' ainda não existe no Model Registry, o notebook vai treinar um baseline e registrar em {catalog}.gold.credito_risco_xgb — isso é esperado, não é erro. Confira se aparece 'Alias '@champion' apontado para a versão N' no final dessa etapa.

5
Rodar de novo e confirmar que carrega do Registry
Rode 02_vies_threshold_roi.py uma segunda vez. Dessa vez deve aparecer 'Modelo carregado do Registry' em vez de treinar de novo — confirma que o alias '@champion' foi setado corretamente e que o notebook está consumindo do Registry, não retreinando a cada execução (isso importa para reprodutibilidade: Q1–Q4 devem usar sempre a mesma versão do modelo).

6
Conferir se o viés de Q3 bate com o README
No output de Q3, confira se os dois alertas do README aparecem: FPR alto na faixa 18-30 (~29,7%) e FNR alto na faixa 60+ (~48,3%). Os números exatos podem variar um pouco (split aleatório diferente), mas a direção do viés deve se repetir — se sumir completamente, vale investigar se os bins de idade/renda ficaram iguais aos do notebook original.

7
Checar que H3 e Q2 dão thresholds diferentes (por design)
Compare o threshold impresso em H3 (01_hipoteses) com o threshold ótimo impresso em Q2 (02_vies). Devem ser DIFERENTES (H3 usa piso de aprovação, Q2 minimiza custo) — se vierem idênticos, algo foi copiado errado. Da mesma forma, o impacto financeiro de H4 e de Q4 podem diferir; isso é esperado, não é bug (ver comentário em src/config/business_params.py).

8
(Opcional) Testar as funções isoladamente com dado sintético
Numa célula solta, importe as funções puras e teste com dados sintéticos, sem depender do cluster/Gold: from src.models.hypothesis_validation import validate_h1, validate_h2 — monte um shap_importance_df de brinquedo com pandas e confira se o status retornado bate com o esperado. Isso valida a lógica de negócio isolada da infraestrutura Databricks.


```